# 03 — Prompt Templates

So far we've written prompts as plain strings. That's fine for a one-off, but real
applications build prompts from **variable data**: a user's question, a document, a
language to translate into. Hard-coding strings and gluing them together with f-strings
gets messy and error-prone fast.

A **prompt template** is a reusable recipe: fixed instructions plus named slots you fill
in at call time. In this notebook:

1. `PromptTemplate` — templating a single string.
2. `ChatPromptTemplate` — templating a multi-message chat prompt.
3. `MessagesPlaceholder` — dropping a whole conversation into a template.
4. **Few-shot** prompting — teaching by example.
5. **Partial** variables.
6. Connecting a template directly to the model with `|` (a preview of LCEL).

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env file first."

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)
print("Model ready.")

Model ready.


## 1. `PromptTemplate` — a single templated string

`PromptTemplate` holds a string with `{placeholders}`. You call `.invoke()` with a dict
to fill them. It returns a `PromptValue` you can convert to a string or messages.

In [2]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "Translate the following text into {language}.\n\nText: {text}"
)

# Fill the slots:
filled = template.invoke({"language": "French", "text": "Good morning, how are you?"})
print(filled.to_string())

Translate the following text into French.

Text: Good morning, how are you?


The template *describes* the prompt; the dict *supplies* the data. You can reuse the
same template with different inputs, and the placeholders document exactly what data the
prompt needs. To actually run it, pass the filled prompt to the model.

In [3]:
prompt_value = template.invoke({"language": "German", "text": "Where is the train station?"})
print(model.invoke(prompt_value).content)

Wo ist der Bahnhof?


## 2. `ChatPromptTemplate` — templating messages

Most of the time you want a structured chat prompt (system + human), not one big string.
`ChatPromptTemplate` templates a list of messages. Each message can contain placeholders,
and you fill them all at once.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains {topic} concepts to {audience}."),
    ("human", "{question}"),
])

messages = prompt.invoke({
    "topic": "machine learning",
    "audience": "high school students",
    "question": "What is overfitting?",
})

# See exactly what gets sent to the model:
for m in messages.to_messages():
    print(f"[{m.type}] {m.content}")

[system] You are a helpful assistant that explains machine learning concepts to high school students.
[human] What is overfitting?


In [5]:
prompt_result = prompt.invoke({
    "topic": "machine learning",
    "audience": "high school students",
    "question": "What is overfitting?",
})
prompt_result

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant that explains machine learning concepts to high school students.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is overfitting?', additional_kwargs={}, response_metadata={})])

In [7]:
# And run it:
print(model.invoke(prompt_result).content)

Imagine you're studying for a big test, and your teacher gives you a set of practice questions.

1.  **A Good Student (Like a Good Machine Learning Model):**
    *   Would try to understand the *concepts* behind each question.
    *   They'd learn the rules, the formulas, and how to apply them.
    *   When the actual test comes, even if the questions are slightly different, they can use their understanding to solve them. They **generalize** well.

2.  **An Overfitting Student (Like an Overfitting Machine Learning Model):**
    *   Instead of understanding the concepts, they just *memorize* the answers to those exact practice questions, word for word.
    *   They know that for question #3, the answer is "B," but they don't know *why*.
    *   Now, the day of the test comes. If the teacher asks the *exact same* questions, this student would do great!
    *   But what if the questions are slightly different, or phrased in another way, even if they're testing the same concept? This stude

## 3. Connect the template to the model with `|`

Calling `prompt.invoke(...)` then `model.invoke(...)` by hand is repetitive. Because both
a prompt and a model are **Runnables**, you can pipe them together with `|` into a single
**chain**. The chain takes the input dict, fills the prompt, and sends it to the model —
in one `.invoke()`.

This is your first taste of **LCEL** (LangChain Expression Language), covered fully in
notebook 05. For now, just appreciate how clean it reads.

In [ ]:
chain = prompt | model

result = chain.invoke({
    "topic": "databases",
    "audience": "beginners",
    "question": "What is an index and why is it fast?",
})
print(result.content)

## 4. `MessagesPlaceholder` — inserting a conversation

A template usually has fixed roles. But what if you want to insert an entire prior
conversation (a variable number of messages) into the prompt? `MessagesPlaceholder`
reserves a slot that you fill with a *list of messages* — perfect for chat history.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant."),
    MessagesPlaceholder("history"),   # a list of messages goes here
    ("human", "{question}"),
])

history = [
    HumanMessage(content="My name is Sara."),
    AIMessage(content="Nice to meet you, Sara!"),
]

chain = prompt | model
print(chain.invoke({"history": history, "question": "What's my name?"}).content)

The model answers "Sara" because the placeholder injected the earlier turns. In
notebook 06 we'll use exactly this pattern to give a chatbot memory.

## 5. Few-shot prompting — teaching by example

Sometimes the best way to specify a task is to *show* examples rather than describe rules.
This is **few-shot prompting**. You include a handful of input/output pairs in the prompt,
and the model imitates the pattern.

Below we teach the model to convert words into a fixed "emoji + definition" format using
two examples, then ask about a new word.

In [8]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"word": "cat", "answer": "cat — a small domesticated feline kept as a pet."},
    {"word": "guitar", "answer": "guitar — a stringed musical instrument played with the fingers or a pick."},
]

# Template for ONE example pair:
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Define: {word}"),
    ("ai", "{answer}"),
])

# Wrap all examples:
few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Final prompt: system + the examples + the real question.
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You define words in the exact format shown by the examples."),
    few_shot,
    ("human", "Define: {word}"),
])

# Inspect what the model will see:
for m in final_prompt.invoke({"word": "telescope"}).to_messages():
    print(f"[{m.type}] {m.content}")

[system] You define words in the exact format shown by the examples.
[human] Define: cat
[ai] cat — a small domesticated feline kept as a pet.
[human] Define: guitar
[ai] guitar — a stringed musical instrument played with the fingers or a pick.
[human] Define: telescope


In [9]:
# Run it — the answer should match the demonstrated format.
chain = final_prompt | model
print(chain.invoke({"word": "telescope"}).content)

telescope — an optical instrument designed to make distant objects appear nearer, larger, or brighter.


> **When to use few-shot.** Reach for it when the *format* or *style* is hard to
> describe but easy to demonstrate, or when the model keeps getting an edge case wrong.
> Each example costs tokens, so use the fewest that reliably work.

## 6. Partial variables

Sometimes part of a prompt is known up front (a fixed role, today's date) and the rest
comes later. `.partial()` pre-fills some variables, returning a new template that only
needs the remaining ones. This is handy for building specialized templates from a general
one.

In [ ]:
base = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Keep answers under 3 sentences."),
    ("human", "{question}"),
])

# Pre-fill the role once:
lawyer_prompt = base.partial(role="contract lawyer")

chain = lawyer_prompt | model
print(chain.invoke({"question": "What is an indemnity clause?"}).content)

## Your turn

Five exercises built around prompts you'd reuse in production — a support-reply template, a
localized copy generator, an intent router, a context-aware help bot, and a docstring writer.
Try each in a blank cell before opening its solution.

**Exercise 1 — Reusable support-reply template.** Build a `ChatPromptTemplate` with slots
for `{tone}`, `{name}`, and `{issue}`, then use `.partial()` to bake in the parts that never
change (your company and agent name). Run it on a sample complaint.

*Sample run (illustrative):*

``` python
# Prompt text can be like:
s = """You write {tone} support replies for {company}. Sign off as '{agent}, {company} Support'.
human: Customer {name} wrote: {issue}
Write a short reply."""
```

```text
Hi Priya,

I'm so sorry your order #1024 arrived with a cracked screen — that's not the experience we
want for you. I've arranged a free replacement that will ship today, and you won't need to
return the damaged unit. Please let me know if there's anything else I can do.

Alex, Northwind Support
```

<details><summary>Show solution</summary>

```python
from langchain_core.prompts import ChatPromptTemplate

reply_tmpl = ChatPromptTemplate.from_messages([
    ("system", "You write {tone} support replies for {company}. Sign off as '{agent}, {company} Support'."),
    ("human", "Customer {name} wrote: {issue}\nWrite a short reply."),
])
my_replies = reply_tmpl.partial(company="Northwind", agent="Alex")   # fixed once

chain = my_replies | model
print(chain.invoke({
    "tone": "warm and apologetic",
    "name": "Priya",
    "issue": "My order #1024 arrived with a cracked screen.",
}).content)
```

`.partial()` turns one general template into a specialized one you can reuse all day.

</details>

**Exercise 2 — Localized marketing copy.** Make a `ChatPromptTemplate` with `{audience}`,
`{language}`, and `{product}` slots, pipe it to the model, and generate the same blurb in
three languages with a loop.

*Sample run (illustrative):*

```text
--- English ---
Dinner sorted in 10 minutes flat. Our healthy meal kits give busy parents wholesome,
kid-approved meals without the planning or the cleanup.

--- Spanish ---
La cena lista en solo 10 minutos. Nuestros kits de comida saludable ofrecen a los padres
ocupados platos nutritivos que encantan a los niños, sin planificar ni limpiar.

--- Japanese ---
夕食はわずか10分で完成。忙しい親御さんに、準備も片付けもいらない、子どもが喜ぶ
ヘルシーなミールキットをお届けします。
```

<details><summary>Show solution</summary>

```python
copy_tmpl = ChatPromptTemplate.from_messages([
    ("system", "You are a marketing copywriter. Write for {audience} in {language}."),
    ("human", "Write a two-sentence product blurb for: {product}"),
])
chain = copy_tmpl | model
for lang in ["English", "Spanish", "Japanese"]:
    print(f"--- {lang} ---")
    print(chain.invoke({
        "audience": "busy parents",
        "language": lang,
        "product": "a 10-minute healthy meal-kit subscription",
    }).content, "\n")
```

</details>

**Exercise 3 — Few-shot intent router.** Use `FewShotChatMessagePromptTemplate` to map a
customer message to one intent — `track_order`, `refund`, or `faq` — by showing three
examples, then route a new message.

*Sample run (illustrative):*

``` text
Like:
Where's my package? --- track_order
I want my money back --- refund
Do you ship to Canada? --- faq
```

<details><summary>Show solution</summary>

```python
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"msg": "Where's my package?",     "intent": "track_order"},
    {"msg": "I want my money back",    "intent": "refund"},
    {"msg": "Do you ship to Canada?",  "intent": "faq"},
]
ex_prompt = ChatPromptTemplate.from_messages([("human", "{msg}"), ("ai", "{intent}")])
few = FewShotChatMessagePromptTemplate(example_prompt=ex_prompt, examples=examples)

router = ChatPromptTemplate.from_messages([
    ("system", "Label the message with one intent: track_order, refund, or faq. Reply with the label only."),
    few,
    ("human", "{msg}"),
])
chain = router | model
print(chain.invoke({"msg": "Can I change the shirt size before it ships?"}).content)
```

The examples pin down the exact label set far more reliably than a written rule.

</details>

**Exercise 4 — Context-aware help bot.** Use `MessagesPlaceholder` to feed prior turns into
a support prompt, so a follow-up question is answered using what the customer already said.

*Sample run (illustrative):*

```text
Yes! Since you're on the Family plan, which supports up to 6 accounts, you can add your
parents — just send them an invite from your account settings.
```

<details><summary>Show solution</summary>

```python
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

bot = ChatPromptTemplate.from_messages([
    ("system", "You are a concise support agent for a music-streaming app."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
history = [
    HumanMessage(content="I'm on the Family plan."),
    AIMessage(content="Got it — the Family plan supports up to 6 accounts."),
]
chain = bot | model
print(chain.invoke({"history": history, "question": "Can I add my parents to it?"}).content)
```

The bot answers in context because the placeholder injected the earlier turns.

</details>

**Exercise 5 — Docstring generator.** Use a single-string `PromptTemplate` with `{style}`,
`{language}`, and `{code}` slots to add a docstring to a function. Run it on a short
Python function.

*Sample run (illustrative):*

```text
def slugify(title):
    """Convert a title into a URL-friendly slug.

    Args:
        title: The text to convert.

    Returns:
        A lowercase, hyphen-separated version of the title.
    """
    return '-'.join(title.lower().split())
```

<details><summary>Show solution</summary>

```python
from langchain_core.prompts import PromptTemplate

doc_tmpl = PromptTemplate.from_template(
    "Add a concise {style} docstring to this {language} function. "
    "Return only the function with the docstring added.\n\n{code}"
)
chain = doc_tmpl | model
code = "def slugify(title):\n    return '-'.join(title.lower().split())"
print(chain.invoke({"style": "Google-style", "language": "Python", "code": code}).content)
```

A single-string `PromptTemplate` is plenty when there's no system/user split to model.

</details>

## Summary

- **`PromptTemplate`** templates a single string; **`ChatPromptTemplate`** templates a
  list of messages — usually what you want.
- Filling a template returns a `PromptValue`; `prompt | model` pipes the filled prompt
  straight into the model as one chain.
- **`MessagesPlaceholder`** injects a variable-length list of messages (e.g. chat history).
- **Few-shot** templates teach a task by example — great for fixing format or style.
- **`.partial()`** pre-fills some variables to specialize a template.

**Next:** [04 — Output Parsers & Structured Output](04_Output_Parsers_and_Structured_Output.ipynb).
We've been reading `.content` strings; now we'll get clean, validated data structures back.